In [35]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import pickle
import plotly.graph_objs as go
import plotly.express as px
import numpy as np
from scipy.spatial import KDTree
import torch.nn.functional as F
from plotly.subplots import make_subplots
import plotly.graph_objs as go
import plotly.express as px
import plotly.subplots as sp

import warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*__floordiv__.*")

### Prepare dataset

In [36]:
with open('../debug.pickle', 'rb') as f:
    data = pickle.load(f)

# Load (1) shape, (2) occupancy, (3) matching feature pairs
src_shape_feats = F.normalize(data['src_shape_feats'].unsqueeze(0), p=2, dim=1).cpu().detach()
trg_shape_feats = F.normalize(data['trg_shape_feats'].unsqueeze(0), p=2, dim=1).cpu().detach()
src_occ_feats = F.normalize(data['src_occ_feats'].unsqueeze(0), p=2, dim=1).cpu().detach()
trg_occ_feats = F.normalize(data['trg_occ_feats'].unsqueeze(0), p=2, dim=1).cpu().detach()

# Load Orientations
src_ori = data['src_ori'].cpu().detach().numpy()
trg_ori = data['trg_ori'].cpu().detach().numpy()
src_gt_rot = data['src_gt_rot'].cpu().detach().numpy()
trg_gt_rot = data['trg_gt_rot'].cpu().detach().numpy()

# Load GT corr and Pred. corr
gt_correspondence = data['gt_correspondence'].cpu().detach()
pred_corr = data['pred_corr'].cpu().detach()

# Load randomly transformed point cloud and translate target pc for better visualization
src_pcd = data['src_pcd'].cpu().detach()
trg_pcd = data['trg_pcd'].cpu().detach()
src_pcd_raw = data['src_pcd_raw'].cpu().detach()
trg_pcd_raw = data['trg_pcd_raw'].cpu().detach()
trg_pcd[:, 2] += 1

### Calculate Cosine Distances

In [24]:
k=1024

# Shape top-k
shape_scores = torch.einsum('b c n , b c m -> b n m', src_shape_feats, trg_shape_feats)
top_k_scores, top_k_indices = torch.topk(shape_scores.contiguous().view(-1), k, largest=True)
top_k_row_indices = top_k_indices // trg_pcd.size(0)
top_k_col_indices = top_k_indices % trg_pcd.size(0)
shape_pred_corr = torch.stack([top_k_row_indices, top_k_col_indices],dim=1)

# Occupancy top-k
occ_scores = torch.einsum('b c n , b c m -> b n m', src_occ_feats, -trg_occ_feats)
top_k_scores, top_k_indices = torch.topk(occ_scores.contiguous().view(-1), k, largest=True)
top_k_row_indices = top_k_indices // trg_pcd.size(0)
top_k_col_indices = top_k_indices % trg_pcd.size(0)
occ_pred_corr = torch.stack([top_k_row_indices, top_k_col_indices],dim=1)

In [31]:
# Step 1: Randomly select an index `i` from `gt_correspondence[:,1]`
# random_idx = gt_correspondence[:, 1][torch.randint(0, gt_correspondence.size(0), (1,))].item()
random_s, random_t = gt_correspondence[torch.randint(0, gt_correspondence.size(0), (1,))][0]
random_s, random_t = random_s.item(), random_t.item()
selected_src_point = src_pcd[random_s]
selected_trg_point = trg_pcd[random_t]
selected_shape_scores = shape_scores[0, :, random_t].numpy()
shape_color_scale = (selected_shape_scores + 1) / 2
selected_occ_scores = occ_scores[0, :, random_t].numpy()
occ_color_scale = (selected_occ_scores + 1) / 2
custom_colorscale = [[0, 'red'], [0.5, 'white'], [1, 'blue']]

# Create a subplot figure with two plots side by side
fig = sp.make_subplots(rows=1, cols=3, subplot_titles=("Target Point Cloud", "Shape Cosine Distance", "Occ Cosine Distance"),
                       specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}, {'type': 'scatter3d'}]])

# Plot target point cloud (trg_pcd) in semi-transparent blue in the first subplot
fig.add_trace(go.Scatter3d(
    x=trg_pcd[:, 0].numpy(),
    y=trg_pcd[:, 1].numpy(),
    z=trg_pcd[:, 2].numpy(),
    mode='markers',
    marker=dict(size=2, color='gray', opacity=0.3),
    name="Target Point Cloud"
), row=1, col=1)

# Highlight selected target point in opaque green in the first subplot
fig.add_trace(go.Scatter3d(
    x=[selected_trg_point[0].item()],
    y=[selected_trg_point[1].item()],
    z=[selected_trg_point[2].item()],
    mode='markers',
    marker=dict(size=4, color='green', opacity=1.0),
    name="Selected Target Point"
), row=1, col=1)

# Highlight selected target point in opaque green in the first subplot
fig.add_trace(go.Scatter3d(
    x=[selected_src_point[0].item()],
    y=[selected_src_point[1].item()],
    z=[selected_src_point[2].item()],
    mode='markers',
    marker=dict(size=4, color='green', opacity=1.0),
    name="Selected Source Point"
), row=1, col=2)
# Highlight selected target point in opaque green in the first subplot
fig.add_trace(go.Scatter3d(
    x=[selected_src_point[0].item()],
    y=[selected_src_point[1].item()],
    z=[selected_src_point[2].item()],
    mode='markers',
    marker=dict(size=4, color='green', opacity=1.0),
    name="Selected Source Point"
), row=1, col=3)

# Plot source point cloud (src_pcd) with interpolated color based on scores in the second subplot
fig.add_trace(go.Scatter3d(
    x=src_pcd[:, 0].numpy(),
    y=src_pcd[:, 1].numpy(),
    z=src_pcd[:, 2].numpy(),
    mode='markers',
    marker=dict(size=2, color=shape_color_scale, colorscale=custom_colorscale, 
                colorbar=dict(title="Score", tickvals=[0, 0.5, 1], ticktext=['-1', '0', '1'])),
    name="Source Point Cloud (Interpolated Scores)"
), row=1, col=2)

# Plot source point cloud (src_pcd) with interpolated color based on scores in the second subplot
fig.add_trace(go.Scatter3d(
    x=src_pcd[:, 0].numpy(),
    y=src_pcd[:, 1].numpy(),
    z=src_pcd[:, 2].numpy(),
    mode='markers',
    marker=dict(size=2, color=occ_color_scale, colorscale=custom_colorscale, 
                colorbar=dict(title="Score", tickvals=[0, 0.5, 1], ticktext=['-1', '0', '1'])),
    name="Source Point Cloud (Interpolated Scores)"
), row=1, col=3)

# Set layout for better visualization
fig.update_layout(title="Separate Views of Target and Source Point Clouds with Custom Color Legend", showlegend=False)

# Resize layout and show
fig.update_layout(width=1000, height=750)
fig.show()

### Shape & Occupancy Descriptor top-k Visualization

In [32]:
# Shape Descriptor top-k visualization
src_corr_points_1 = src_pcd[shape_pred_corr[:, 0]]
trg_corr_points_1 = trg_pcd[shape_pred_corr[:, 1]]

# Occupancy Descriptor top-k visualization
src_corr_points_2 = src_pcd[occ_pred_corr[:, 0]]
trg_corr_points_2 = trg_pcd[occ_pred_corr[:, 1]]

# Subplots
fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]],
                    subplot_titles=("Shape Descriptor", "Occupancy Descriptor"))

####### Shape Plot #######
fig.add_trace(go.Scatter3d(
    x=src_pcd[:, 0].numpy(), y=src_pcd[:, 1].numpy(), z=src_pcd[:, 2].numpy(),
    mode='markers', marker=dict(size=2, color='red'),# name='Source Point Cloud'
), row=1, col=1)
fig.add_trace(go.Scatter3d(
    x=trg_pcd[:, 0].numpy(), y=trg_pcd[:, 1].numpy(), z=trg_pcd[:, 2].numpy(),
    mode='markers', marker=dict(size=2, color='blue'),# name='Target Point Cloud'
), row=1, col=1)
fig.add_trace(go.Scatter3d(
    x=src_corr_points_1[:, 0].numpy(), y=src_corr_points_1[:, 1].numpy(), z=src_corr_points_1[:, 2].numpy(),
    mode='markers', marker=dict(size=5, color='green'),# name='Source Correspondences'
), row=1, col=1)
fig.add_trace(go.Scatter3d(
    x=trg_corr_points_1[:, 0].numpy(), y=trg_corr_points_1[:, 1].numpy(), z=trg_corr_points_1[:, 2].numpy(),
    mode='markers', marker=dict(size=5, color='green'),# name='Target Correspondences'
), row=1, col=1)

for i in range(shape_pred_corr.size(0)):
    fig.add_trace(go.Scatter3d(
        x=[src_corr_points_1[i, 0].item(), trg_corr_points_1[i, 0].item()],
        y=[src_corr_points_1[i, 1].item(), trg_corr_points_1[i, 1].item()],
        z=[src_corr_points_1[i, 2].item(), trg_corr_points_1[i, 2].item()],
        mode='lines', line=dict(color='black', width=2), showlegend=False
    ), row=1, col=1)
####### Shape Plot #######

####### Occupancy Plot #######
fig.add_trace(go.Scatter3d(
    x=src_pcd[:, 0].numpy(), y=src_pcd[:, 1].numpy(), z=src_pcd[:, 2].numpy(),
    mode='markers', marker=dict(size=2, color='red'),# name='Source Point Cloud'
), row=1, col=2)
fig.add_trace(go.Scatter3d(
    x=trg_pcd[:, 0].numpy(), y=trg_pcd[:, 1].numpy(), z=trg_pcd[:, 2].numpy(),
    mode='markers', marker=dict(size=2, color='blue'),# name='Target Point Cloud'
), row=1, col=2)
fig.add_trace(go.Scatter3d(
    x=src_corr_points_2[:, 0].numpy(), y=src_corr_points_2[:, 1].numpy(), z=src_corr_points_2[:, 2].numpy(),
    mode='markers', marker=dict(size=5, color='green'),# name='Source Correspondences'
), row=1, col=2)
fig.add_trace(go.Scatter3d(
    x=trg_corr_points_2[:, 0].numpy(), y=trg_corr_points_2[:, 1].numpy(), z=trg_corr_points_2[:, 2].numpy(),
    mode='markers', marker=dict(size=5, color='green'),# name='Target Correspondences'
), row=1, col=2)

for i in range(occ_pred_corr.size(0)):
    fig.add_trace(go.Scatter3d(
        x=[src_corr_points_2[i, 0].item(), trg_corr_points_2[i, 0].item()],
        y=[src_corr_points_2[i, 1].item(), trg_corr_points_2[i, 1].item()],
        z=[src_corr_points_2[i, 2].item(), trg_corr_points_2[i, 2].item()],
        mode='lines', line=dict(color='black', width=2), showlegend=False
    ), row=1, col=2)
####### Occupancy Plot #######

# Resize layout and show
fig.update_layout(width=1000, height=750)
fig.show()

In [37]:
src_pcd = src_pcd_raw
trg_pcd = trg_pcd_raw

src_ori = src_ori @ src_gt_rot
# norms = np.linalg.norm(src_ori.mean(axis=2), axis=1, keepdims=True)
src_vec = src_ori[:,0]# src_ori.mean(axis=2) / norms

trg_ori = trg_ori @ trg_gt_rot
# norms = np.linalg.norm(trg_ori.mean(axis=2), axis=1, keepdims=True)
trg_vec = trg_ori[:,0]# trg_ori.mean(axis=2) / norms

dot_size = 0.5
# 첫 번째 포인트 클라우드 (src_pcd)
trace1 = go.Scatter3d(
    x=src_pcd[:, 0],
    y=src_pcd[:, 1],
    z=src_pcd[:, 2],
    mode='markers',
    marker=dict(
        size=dot_size,
        color='blue',  # 포인트 색상
        opacity=0.8
    ),
    name='Source Point Cloud'
)

# 두 번째 포인트 클라우드 (trg_pcd)
trace2 = go.Scatter3d(
    x=trg_pcd[:, 0],
    y=trg_pcd[:, 1],
    z=trg_pcd[:, 2],
    mode='markers',
    marker=dict(
        size=dot_size,
        color='red',  # 포인트 색상
        opacity=0.8
    ),
    name='Target Point Cloud'
)
# 첫 번째 포인트 클라우드에 대한 화살표 (src_pcd의 화살표)
cones1 = go.Cone(
    x=src_pcd[gt_correspondence[:,0], 0],
    y=src_pcd[gt_correspondence[:,0], 1],
    z=src_pcd[gt_correspondence[:,0], 2],
    u=src_vec[gt_correspondence[:,0], 0],
    v=src_vec[gt_correspondence[:,0], 1],
    w=src_vec[gt_correspondence[:,0], 2],
    colorscale=[[0, 'blue'], [1, 'blue']],
    sizemode='absolute',
    sizeref=1,  # 화살표 크기 조절
    anchor='tail',  # 화살표 시작점을 포인트로 설정
    showscale=False,
    name='Source Orientation'
)

# 두 번째 포인트 클라우드에 대한 화살표 (trg_pcd의 화살표)
cones2 = go.Cone(
    x=trg_pcd[gt_correspondence[:,1], 0],
    y=trg_pcd[gt_correspondence[:,1], 1],
    z=trg_pcd[gt_correspondence[:,1], 2],
    u=trg_vec[gt_correspondence[:,1], 0],
    v=trg_vec[gt_correspondence[:,1], 1],
    w=trg_vec[gt_correspondence[:,1], 2],
    colorscale=[[0, 'red'], [1, 'red']],
    sizemode='absolute',
    sizeref=2,  # 화살표 크기 조절
    anchor='tail',  # 화살표 시작점을 포인트로 설정
    showscale=False,
    name='Target Orientation'
)

# 레이아웃 설정
layout = go.Layout(
    title="3D Point Cloud with Orientation Arrows",
    scene=dict(
        xaxis_title='X Axis',
        yaxis_title='Y Axis',
        zaxis_title='Z Axis'
    ),
    width=1000,
    height=1000
)

# 포인트 클라우드와 화살표를 함께 표시
fig = go.Figure(data=[trace1, trace2, cones1, cones2], layout=layout)

# 그래프 출력
fig.show()


In [ ]:
def _is_trg_larger(src_pcd, trg_pcd):
    src_volume = np.prod(src_pcd.max(axis=0) - src_pcd.min(axis=0))
    trg_volume = np.prod(trg_pcd.max(axis=0) - trg_pcd.min(axis=0))

    return src_volume < trg_volume

def _pairwise_mating(src_pcd, trg_pcd, rotat, trans, is_trg_larger):
    pcd_t = []
    if is_trg_larger:
        src_pcd_t = _transform(src_pcd, rotat, -trans, True)
        pcd_t = [src_pcd_t, trg_pcd]
    else:
        trg_pcd_t = _transform(trg_pcd, np.linalg.inv(rotat), trans, False)
        pcd_t = [src_pcd, trg_pcd_t]
    return np.concatenate(pcd_t, axis=0), pcd_t

def _transform(pcd, rotat=None, trans=None, rotate_first=True):
    if rotat is None:
        rotat = np.eye(3)
    if trans is None:
        trans = np.zeros(3)

    if rotate_first:
        return np.dot(pcd, rotat.T) + trans
    else:
        return np.dot(pcd + trans, rotat.T)

In [ ]:
is_trg_larger = _is_trg_larger(src_pcd, trg_pcd)

# Assemble using prediction, pseudo-gt, and ground-truth
assm_pred, pcds_pred = _pairwise_mating(src_pcd, trg_pcd, estimated_rotat, estimated_trans, is_trg_larger)

In [ ]:
import plotly.graph_objs as go
import plotly.express as px
dot_size = 0.5
src_pcd = pcds_pred[0]
trg_pcd = pcds_pred[1]
# 첫 번째 포인트 클라우드 (src_pcd)
trace1 = go.Scatter3d(
    x=src_pcd[:, 0],
    y=src_pcd[:, 1],
    z=src_pcd[:, 2],
    mode='markers',
    marker=dict(
        size=dot_size,
        color='blue',  # 포인트 색상
        opacity=0.8
    ),
    name='Source Point Cloud'
)

# 두 번째 포인트 클라우드 (trg_pcd)
trace2 = go.Scatter3d(
    x=trg_pcd[:, 0],
    y=trg_pcd[:, 1],
    z=trg_pcd[:, 2],
    mode='markers',
    marker=dict(
        size=dot_size,
        color='red',  # 포인트 색상
        opacity=0.8
    ),
    name='Target Point Cloud'
)
# 첫 번째 포인트 클라우드에 대한 화살표 (src_pcd의 화살표)
cones1 = go.Cone(
    x=src_pcd[gt_correspondence[:,0], 0],
    y=src_pcd[gt_correspondence[:,0], 1],
    z=src_pcd[gt_correspondence[:,0], 2],
    u=src_vec[gt_correspondence[:,0], 0],
    v=src_vec[gt_correspondence[:,0], 1],
    w=src_vec[gt_correspondence[:,0], 2],
    colorscale=[[0, 'red'], [1, 'red']],
    sizemode='absolute',
    sizeref=1,  # 화살표 크기 조절
    anchor='tail',  # 화살표 시작점을 포인트로 설정
    showscale=False,
    name='Source Orientation'
)

# 두 번째 포인트 클라우드에 대한 화살표 (trg_pcd의 화살표)
cones2 = go.Cone(
    x=trg_pcd[gt_correspondence[:,1], 0],
    y=trg_pcd[gt_correspondence[:,1], 1],
    z=trg_pcd[gt_correspondence[:,1], 2],
    u=trg_vec[gt_correspondence[:,1], 0],
    v=trg_vec[gt_correspondence[:,1], 1],
    w=trg_vec[gt_correspondence[:,1], 2],
    colorscale=[[0, 'blue'], [1, 'blue']],
    sizemode='absolute',
    sizeref=2,  # 화살표 크기 조절
    anchor='tail',  # 화살표 시작점을 포인트로 설정
    showscale=False,
    name='Target Orientation'
)

# 레이아웃 설정
layout = go.Layout(
    title="3D Point Cloud with Orientation Arrows",
    scene=dict(
        xaxis_title='X Axis',
        yaxis_title='Y Axis',
        zaxis_title='Z Axis'
    ),
    width=1000,
    height=1000
)

# 포인트 클라우드와 화살표를 함께 표시
fig = go.Figure(data=[trace1, trace2, cones1, cones2], layout=layout)

# 그래프 출력
fig.show()


In [ ]:
src_ori

In [ ]:
trg_ori

In [ ]:
print(src_ori.shape)
print(trg_ori.shape)